# Superstore Sales Analysis
## Subqueries | CTEs | Window Functions

**Dataset:** Sample Superstore — 9,994 rows (2014–2017)  
**Engine:** DuckDB (standard SQL)  
**Tables:** `superstore_raw` → `customers`, `orders`, `products`


## Step 1: Setup — Load Data & Create Tables

In [ ]:
import pandas as pd
import duckdb

# Connect and load the dataset
con = duckdb.connect("superstore.db")

con.execute("""
CREATE OR REPLACE TABLE superstore_raw AS
SELECT * FROM read_csv_auto('superstore.csv', header=True)
""")

count = con.execute("SELECT COUNT(*) FROM superstore_raw").fetchone()[0]
cols  = [r[0] for r in con.execute("DESCRIBE superstore_raw").fetchall()]
print(f"superstore_raw loaded: {count} rows")
print(f"Columns ({len(cols)}):", cols)


superstore_raw loaded: 9994 rows
Columns (21): ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [ ]:
# Create 3 normalised tables using SELECT DISTINCT

con.execute("""
CREATE OR REPLACE TABLE customers AS
SELECT DISTINCT
    "Customer ID"   AS customer_id,
    "Customer Name" AS customer_name,
    "Segment"       AS segment,
    "Region"        AS region
FROM superstore_raw
""")

con.execute("""
CREATE OR REPLACE TABLE products AS
SELECT DISTINCT
    "Product ID"   AS product_id,
    "Product Name" AS product_name,
    "Category"     AS category,
    "Sub-Category" AS sub_category
FROM superstore_raw
""")

con.execute("""
CREATE OR REPLACE TABLE orders AS
SELECT
    "Row ID"      AS row_id,
    "Order ID"    AS order_id,
    "Order Date"  AS order_date,
    "Ship Date"   AS ship_date,
    "Ship Mode"   AS ship_mode,
    "Customer ID" AS customer_id,
    "Product ID"  AS product_id,
    "Sales"       AS sales,
    "Quantity"    AS quantity,
    "Discount"    AS discount,
    "Profit"      AS profit
FROM superstore_raw
""")

for tbl in ['customers', 'products', 'orders']:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"  {tbl:12s}: {n:,} rows")


  customers   : 793 rows
  products    : 1,894 rows
  orders      : 9,994 rows


## Step 2: Required Queries

### Query 1 — Orders Where Sales > Average Sales
**Technique:** Subquery

The inner subquery computes `AVG(sales)` across all orders. The outer query filters rows where individual order sales exceed that average.

In [ ]:
q1 = con.execute("""
SELECT
    order_id,
    customer_id,
    ROUND(sales, 2) AS sales
FROM orders
WHERE sales > (
    SELECT AVG(sales)
    FROM orders
)
ORDER BY sales DESC
LIMIT 10
""").df()
print(q1.to_string(index=False))


        order_id customer_id    sales
  CA-2014-145317    SM-20320 22638.48
  CA-2016-118689    TC-20980 17499.95
  CA-2017-140151    RB-19360 13999.96
  CA-2017-127180    TA-21385 11199.97
  CA-2017-166709    HL-15040 10499.97
  CA-2016-117121    AB-10105  9892.74
  CA-2014-116904    SC-20095  9449.95
  US-2016-107440    BS-11365  9099.93
  CA-2016-158841    SE-20110  8749.95
  CA-2016-143714    CC-12370  8399.98


### Query 2 — Highest Sales Order for Each Customer
**Technique:** Correlated Subquery

For each row in `orders`, the correlated subquery checks whether `sales` equals the maximum sales for that specific customer. Only the peak order per customer is returned.

In [ ]:
q2 = con.execute("""
SELECT
    o.customer_id,
    c.customer_name,
    o.order_id,
    ROUND(o.sales, 2) AS highest_order_sales
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
LIMIT 10
""").df()
print(q2.to_string(index=False))


 customer_id     customer_name       order_id  highest_order_sales
    SM-20320       Sean Miller  CA-2014-145317             22638.48
    TC-20980      Tamara Chand  CA-2016-118689             17499.95
    RB-19360      Raymond Buch  CA-2017-140151             13999.96
    TA-21385  Tracy Andreadi    CA-2017-127180             11199.97
    HL-15040   Hunter Lonsdale  CA-2017-166709             10499.97
    AB-10105     Adrian Barton  CA-2016-117121              9892.74
    SC-20095      Sanjit Chand  CA-2014-116904              9449.95
    BS-11365      Bill Shonely  US-2016-107440              9099.93
    SE-20110      Sanjit Engle  CA-2016-158841              8749.95
    CC-12370 Christopher Conant CA-2016-143714              8399.98


### Query 3 — Total Sales per Customer
**Technique:** CTE

A CTE named `customer_sales` aggregates all order-level sales per customer. The main query reads from the CTE like a regular table.

In [ ]:
q3 = con.execute("""
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT * FROM customer_sales
ORDER BY total_sales DESC
LIMIT 10
""").df()
print(q3.to_string(index=False))


 customer_id      customer_name  total_sales
    SM-20320        Sean Miller    100172.20
    TC-20980       Tamara Chand     76208.87
    RB-19360       Raymond Buch     60469.36
    AB-10105      Adrian Barton     57894.28
    SC-20095       Sanjit Chand     56569.34
    SE-20110       Sanjit Engle     48837.75
    CC-12370 Christopher Conant     48516.29
    SV-20365        Seth Vernon     45883.80
    CJ-12010    Caroline Jumper     44659.90
    CL-12565        Clay Ludtke     43522.18
(showing top 10 of 793 customers)


### Query 4 — Customers Whose Total Sales Are Above Average
**Technique:** CTE + Subquery

The CTE computes per-customer totals. A subquery inside the WHERE clause then calculates the average of those totals, and only customers exceeding it are returned.

In [ ]:
q4 = con.execute("""
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT customer_id, customer_name, total_sales
FROM customer_sales
WHERE total_sales > (
    SELECT AVG(total_sales) FROM customer_sales
)
ORDER BY total_sales DESC
""").df()
print(f"Customers above average: {len(q4)}")
print(q4.head(10).to_string(index=False))


Customers above average: 108
 customer_id      customer_name  total_sales
    SM-20320        Sean Miller    100172.20
    TC-20980       Tamara Chand     76208.87
    RB-19360       Raymond Buch     60469.36
    AB-10105      Adrian Barton     57894.28
    SC-20095       Sanjit Chand     56569.34
    SE-20110       Sanjit Engle     48837.75
    CC-12370 Christopher Conant     48516.29
    SV-20365        Seth Vernon     45883.80
    CJ-12010    Caroline Jumper     44659.90
    CL-12565        Clay Ludtke     43522.18
(108 of 793 customers shown; displaying top 10)


### Query 5 — Rank All Customers by Total Sales
**Technique:** Window Function — `RANK()`

`RANK()` assigns the same rank to ties and skips the next rank. This preserves all customer rows while adding a computed ranking column.

In [ ]:
q5 = con.execute("""
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT
    customer_id,
    customer_name,
    total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM customer_sales
ORDER BY sales_rank
LIMIT 10
""").df()
print(q5.to_string(index=False))


 customer_id      customer_name  total_sales  sales_rank
    SM-20320        Sean Miller    100172.20           1
    TC-20980       Tamara Chand     76208.87           2
    RB-19360       Raymond Buch     60469.36           3
    AB-10105      Adrian Barton     57894.28           4
    SC-20095       Sanjit Chand     56569.34           5
    SE-20110       Sanjit Engle     48837.75           6
    CC-12370 Christopher Conant     48516.29           7
    SV-20365        Seth Vernon     45883.80           8
    CJ-12010    Caroline Jumper     44659.90           9
    CL-12565        Clay Ludtke     43522.18          10
(showing top 10 of 793 ranked customers)


### Query 6 — Row Numbers per Order Within Each Customer
**Technique:** Window Function — `ROW_NUMBER()` + `PARTITION BY`

`PARTITION BY customer_id` resets the counter for each customer. `ORDER BY order_date` sequences their orders chronologically, so row 1 = first ever order for that customer.

In [ ]:
q6 = con.execute("""
SELECT
    order_id,
    customer_id,
    order_date,
    ROUND(sales, 2) AS sales,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS order_row_num
FROM orders
ORDER BY customer_id, order_row_num
LIMIT 12
""").df()
print(q6.to_string(index=False))


        order_id customer_id  order_date   sales  order_row_num
  CA-2015-121391    AA-10315  2015-04-10   26.96              1
  CA-2016-103982    AA-10315  2016-03-03 3930.07              2
  CA-2016-103982    AA-10315  2016-03-03    2.30              3
  CA-2016-103982    AA-10315  2016-03-03  431.98              4
  CA-2016-103982    AA-10315  2016-03-03   41.72              5
  CA-2014-128055    AA-10315  2014-03-31  673.57              6
  CA-2014-128055    AA-10315  2014-03-31   52.98              7
  CA-2017-147039    AA-10315  2017-06-29  362.94              8
  CA-2017-147039    AA-10315  2017-06-29   11.54              9
  CA-2014-138100    AA-10315  2014-09-15   14.94             10
  CA-2014-138100    AA-10315  2014-09-15   14.56             11
  CA-2014-130729    AA-10375  2014-10-24   34.27              1


### Query 7 — Top 3 Customers by Total Sales
**Technique:** Window Function — `DENSE_RANK()`

`DENSE_RANK()` avoids gaps in rank numbers (unlike `RANK()`). Wrapping it in a CTE and filtering `WHERE rnk <= 3` cleanly extracts the top 3.

In [ ]:
q7 = con.execute("""
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
),
ranked AS (
    SELECT
        customer_id,
        customer_name,
        total_sales,
        DENSE_RANK() OVER (ORDER BY total_sales DESC) AS rnk
    FROM customer_sales
)
SELECT customer_id, customer_name, total_sales, rnk AS rank
FROM ranked
WHERE rnk <= 3
ORDER BY rnk
""").df()
print(q7.to_string(index=False))


 customer_id customer_name  total_sales  rank
    SM-20320   Sean Miller    100172.20     1
    TC-20980  Tamara Chand     76208.87     2
    RB-19360  Raymond Buch     60469.36     3


## Step 3: Final Combined Query
**Customer Name | Total Sales | Rank**  
**Technique:** JOIN + CTE + Window Function together


In [ ]:
final = con.execute("""
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c
        ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT
    customer_name            AS "Customer Name",
    total_sales              AS "Total Sales",
    RANK() OVER (
        ORDER BY total_sales DESC
    )                        AS "Rank"
FROM customer_sales
ORDER BY "Rank"
""").df()
print(final.head(20).to_string(index=False))
print(f"\n... ({len(final)} customers total)")


      Customer Name  Total Sales  Rank
        Sean Miller    100172.20     1
       Tamara Chand     76208.87     2
       Raymond Buch     60469.36     3
      Adrian Barton     57894.28     4
       Sanjit Chand     56569.34     5
       Sanjit Engle     48837.75     6
 Christopher Conant     48516.29     7
        Seth Vernon     45883.80     8
    Caroline Jumper     44659.90     9
        Clay Ludtke     43522.18    10
       Ken Lonsdale     42525.69    11
     Karen Ferguson     42417.06    12
       Bill Shonely     42006.61    13
           John Lee     39199.69    14
       Hunter Lopez     38619.89    15
       Tom Ashbrook     38153.50    16
      Jane Waco       37760.13    17
    Patrick O'Brill     37359.36    18
   Chloris Kastensmid     36781.82    19
      Ted Butterfield     35689.77    20

... (793 customers total)


## Mini Project: Customer Sales Insights

### Mini Q1 — Who Are the Top 5 Customers?

In [ ]:
m1 = con.execute("""
WITH customer_sales AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT customer_name, total_sales,
       RANK() OVER (ORDER BY total_sales DESC) AS rank
FROM customer_sales
ORDER BY total_sales DESC
LIMIT 5
""").df()
print(m1.to_string(index=False))


  customer_name  total_sales  rank
    Sean Miller    100172.20     1
   Tamara Chand     76208.87     2
   Raymond Buch     60469.36     3
  Adrian Barton     57894.28     4
   Sanjit Chand     56569.34     5


### Mini Q2 — Who Are the Bottom 5 Customers?

In [ ]:
m2 = con.execute("""
WITH customer_sales AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT customer_name, total_sales,
       RANK() OVER (ORDER BY total_sales ASC) AS rank_from_bottom
FROM customer_sales
ORDER BY total_sales ASC
LIMIT 5
""").df()
print(m2.to_string(index=False))


    customer_name  total_sales  rank_from_bottom
     Lela Donovan         5.30                 1
    Thais Sissman         9.67                 2
     Carl Jackson        16.52                 3
  Mitch Gastineau        16.74                 4
       Roy Skaria        44.66                 5


### Mini Q3 — Which Customers Made Only One Order?

In [ ]:
m3 = con.execute("""
SELECT
    c.customer_name,
    COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_name
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name
""").df()
print(f"Total customers with only 1 order: {len(m3)}")
print(m3.to_string(index=False))


Total customers with only 1 order: 12
        customer_name  order_count
       Anemone Ratner            1
   Anthony O'Donnell            1
         Carl Jackson            1
         Jenna Caffey            1
       Jocasta Rupert            1
         Lela Donovan            1
      Mitch Gastineau            1
    Patricia Hirasaki            1
      Ricardo Emerson            1
        Roland Murray            1
           Roy Skaria            1
    Thais Sissman               1


### Mini Q4 — Which Customers Have Above-Average Sales?

In [ ]:
m4 = con.execute("""
WITH customer_sales AS (
    SELECT o.customer_id, c.customer_name,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT customer_name, total_sales
FROM customer_sales
WHERE total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY total_sales DESC
""").df()
print(f"Customers with above-average sales: {len(m4)} out of 793")
print(m4.head(10).to_string(index=False))


Customers with above-average sales: 108 out of 793
      customer_name  total_sales
        Sean Miller    100172.20
       Tamara Chand     76208.87
       Raymond Buch     60469.36
      Adrian Barton     57894.28
       Sanjit Chand     56569.34
       Sanjit Engle     48837.75
 Christopher Conant     48516.29
        Seth Vernon     45883.80
    Caroline Jumper     44659.90
        Clay Ludtke     43522.18
(displaying top 10 of 108)


### Mini Q5 — What Is the Highest Order Value per Customer?

In [ ]:
m5 = con.execute("""
SELECT
    c.customer_name,
    o.order_id,
    ROUND(o.sales, 2) AS highest_order_value
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY highest_order_value DESC
LIMIT 15
""").df()
print(m5.to_string(index=False))


       customer_name       order_id  highest_order_value
         Sean Miller  CA-2014-145317             22638.48
        Tamara Chand  CA-2016-118689             17499.95
        Raymond Buch  CA-2017-140151             13999.96
    Tracy Andreadi    CA-2017-127180             11199.97
     Hunter Lonsdale  CA-2017-166709             10499.97
       Adrian Barton  CA-2016-117121              9892.74
        Sanjit Chand  CA-2014-116904              9449.95
        Bill Shonely  US-2016-107440              9099.93
        Sanjit Engle  CA-2016-158841              8749.95
  Christopher Conant  CA-2016-143714              8399.98
         Seth Vernon  CA-2014-141817              7999.98
       Caroline Jumper CA-2016-116862             7399.98
         Clay Ludtke  CA-2016-121755              6999.98
        Ken Lonsdale  CA-2015-124906              6799.95
      Karen Ferguson  CA-2016-135427              6599.95


## Key Business Insights

| # | Insight |
|---|--------|
| 1 | **Sean Miller** is the top customer with **$100,172** in total sales — over 31% more than second-placed Tamara Chand ($76,208). |
| 2 | **108 out of 793 customers** (≈13.6%) are above the average total-sales threshold, showing a small high-value customer segment driving disproportionate revenue. |
| 3 | **12 customers** placed only a single order, representing an opportunity for re-engagement campaigns. |
| 4 | **Bottom 5 customers** have cumulative sales under $50 — Lela Donovan at just $5.30. These are likely one-time low-value transactions. |
| 5 | **Sean Miller's single highest order** ($22,638) is more than double the next best order ($17,499 by Tamara Chand), indicating a high-ticket purchase. |
| 6 | Window functions (`RANK`, `ROW_NUMBER`, `DENSE_RANK`) add analytical columns **without collapsing rows**, making them far more powerful than a plain `GROUP BY` for ranking and sequencing tasks. |


## Cleanup

In [ ]:
con.close()
print('Connection closed.')

Connection closed.
